# Crossover operators in DESDEO

Crossover is the step in an evolutionary algorithm that turns (usually) two *parent*
solutions into one or more *offsprings*. It is the main source of exploration in an EA.
Mutation nudges a single solution and introduces small changes, crossover recombines 
information that two different solutions have discovered.

DESDEO implements nine crossover operators. They all expose the same interface,
so swapping one for another is a one-line change, as long as the problem is compatible with the operator.
Each operator currently is designed to handle, e.g., either a problem with all continuous variables, all binary
variables, all integer variables or mixed integer/continuous variables.

This page is about what these operators do, shown through the distribution of
the offspring they generate based on specific parent populations.
The implementation details of these operators are presented at the end of this page.

Related reading:

- [Evolutionary Algorithms in DESDEO](../templates_and_pub_sub) — how operators are assembled into an algorithm.
- [Pydantic interface](../pydantic_interface) and [Using evolutionary algorithms](../../howtoguides/ea_options) — how to configure an operator without writing code.

In [39]:
"""Setup: helpers used by every figure on this page."""
# ruff: noqa: RUF001  (Greek letters are deliberate mathematical notation in the chart labels)

import numpy as np
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

from desdeo.emo.operators.crossover import (
    BlendAlphaCrossover,
    BoundedExponentialCrossover,
    CompositeCrossover,
    LocalCrossover,
    SimulatedBinaryCrossover,
    SingleArithmeticCrossover,
    SinglePointBinaryCrossover,
    UniformIntegerCrossover,
)
from desdeo.emo.operators.evaluator import EMOEvaluator
from desdeo.emo.operators.generator import RandomGenerator
from desdeo.problem.schema import Objective, Problem, Variable, VariableTypeEnum
from desdeo.tools.patterns import Publisher

# --- a throwaway problem to give the operators variable bounds and types ------


def demo_problem(n_vars: int = 2, kind: str = "real", lb: float = 0.0, ub: float = 10.0) -> Problem:
    """A minimal problem whose only job is to carry `n_vars` bounds of a given type."""
    variable_type = {
        "real": VariableTypeEnum.real,
        "integer": VariableTypeEnum.integer,
        "binary": VariableTypeEnum.binary,
    }[kind]
    variables = [
        Variable(
            name=f"x_{i}",
            symbol=f"x_{i}",
            variable_type=variable_type,
            lowerbound=lb,
            upperbound=ub,
            initial_value=lb,
        )
        for i in range(1, n_vars + 1)
    ]
    return Problem(
        name="demo",
        description="Carrier problem for the crossover demonstrations.",
        variables=variables,
        objectives=[
            Objective(
                name="f_1",
                symbol="f_1",
                func=" + ".join(f"x_{i}" for i in range(1, n_vars + 1)),
                maximize=False,
            )
        ],
    )


PROBLEM = demo_problem(2, "real", 0.0, 10.0)


def offspring_cloud(operator, parent_a, parent_b, n_pairs=80000, problem=PROBLEM):
    """Mate the same parent pair `n_pairs` times and return the offspring as an array.

    `to_mate` is honoured verbatim and parents are paired consecutively, so laying the
    population out as [a, b, a, b, ...] and mating the whole thing gives `n_pairs`
    independent draws from the operator for one fixed pair of parents.
    """
    symbols = [v.symbol for v in problem.get_flattened_variables()]
    population = pl.DataFrame(np.tile(np.array([parent_a, parent_b], dtype=float), (n_pairs, 1)), schema=symbols)
    return operator.do(population=population, to_mate=list(range(2 * n_pairs))).to_numpy()


# --- plotting ------------------------------------------------------------------
# Categorical hues (operator identity) and a single-hue ordinal ramp (parameter
# sweeps, where the series are ordered by magnitude rather than merely different).
IDENTITY = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#184f95"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e8e7e3"


def density(values, lo, hi, bins=160):
    """Histogram of `values` as a density curve over [lo, hi].

    Values outside the window are *dropped*, not clipped: clipping would pile every
    outlier into the edge bins and invent a spike that the operator never produced.
    """
    counts, edges = np.histogram(values, bins=bins, range=(lo, hi), density=True)
    return 0.5 * (edges[:-1] + edges[1:]), counts


def style(fig, title, xlabel, ylabel, height=380, showlegend=True):
    """Apply the shared chart styling: light surface, recessive grid, ink text."""
    fig.update_layout(
        title={"text": title, "font": {"size": 15, "color": INK}, "x": 0, "xanchor": "left"},
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font={"color": INK_SOFT, "size": 12},
        height=height,
        margin={"l": 60, "r": 30, "t": 60, "b": 50},
        showlegend=showlegend,
        legend={"orientation": "h", "y": -0.18, "x": 0, "font": {"color": INK_SOFT}},
        hovermode="x unified",
    )
    fig.update_xaxes(title_text=xlabel, gridcolor=GRID, zeroline=False, linecolor=GRID, ticks="outside")
    fig.update_yaxes(title_text=ylabel, gridcolor=GRID, zeroline=False, linecolor=GRID, ticks="outside")
    return fig


def mark_parents(fig, positions, labels=("parent A", "parent B"), row=None, col=None):
    """Dashed reference lines showing where the parents sit."""
    for position, label in zip(positions, labels, strict=False):
        fig.add_vline(
            x=position,
            line={"color": INK_SOFT, "width": 1, "dash": "dot"},
            annotation={"text": label, "font": {"size": 10, "color": INK_SOFT}},
            annotation_position="top",
            row=row,
            col=col,
        )
    return fig


def show(fig):
    # "notebook_connected" pulls plotly.js from the CDN; the plain "notebook" renderer
    # inlines a ~5 MB copy of the library into every single figure.
    fig.show(renderer="notebook_connected", include_plotlyjs="cdn")


# The two parents used throughout: both variables at 3.0 and at 7.0, bounds [0, 10].
PARENT_A, PARENT_B = [3.0, 3.0], [7.0, 7.0]

## Comparison at a glance

| Operator | Variable type | Parameters | Where the offspring land | Inside bounds |
|---|---|---|---|---|
| `SimulatedBinaryCrossover` | continuous | `xover_distribution` (η), `xover_probability`, `uniform_xover_probability`, `bounded` | close to **both** parents; polynomial density in a spread factor β | only if `bounded=True` |
| `BlendAlphaCrossover` | continuous | `alpha`, `repeats`, `sample_each_component` | uniformly over the parent interval, widened by α | yes |
| `LocalCrossover` | continuous | — | uniformly **between** the parents, fresh weight per variable | yes |
| `SingleArithmeticCrossover` | continuous | `xover_probability` | one gene set to the parent mean, every other gene copied | yes |
| `BoundedExponentialCrossover` | continuous | `lambda_` (λ), `xover_probability` | around **its own** parent, exponential decay, truncated at the bounds | yes |
| `SinglePointBinaryCrossover` | binary | — | contiguous blocks swapped at one cut point | n/a |
| `UniformIntegerCrossover` | integer | — | each gene independently taken from either parent | n/a |
| `UniformMixedIntegerCrossover` | mixed | — | as above, without rounding the continuous genes | n/a |
| `CompositeCrossover` | inherited | `operators` | delegates to sub-operators in round-robin | inherited |


### Behaviour of operators for continuous variables

Both parents below have every variable at $3.0$ and $7.0$ respectively, on the box
$[0, 10]^2$. Each panel shows the distribution of 80000 offspring of the first
variable, drawn from that same pair of parents over and over.

In [40]:
setups = [
    (
        "Simulated binary (η=15)",
        lambda: SimulatedBinaryCrossover(
            problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, xover_distribution=15
        ),
    ),
    (
        "Bounded exponential (λ=0.25)",
        lambda: BoundedExponentialCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, lambda_=0.25),
    ),
    (
        "Blend alpha (α=0.5)",
        lambda: BlendAlphaCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, alpha=0.5),
    ),
    ("Local", lambda: LocalCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0)),
    (
        "Single arithmetic",
        lambda: SingleArithmeticCrossover(
            problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, xover_probability=1.0
        ),
    ),
]

fig = make_subplots(
    rows=len(setups),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.045,
    subplot_titles=[name for name, _ in setups],
)
for row, ((name, make_operator), colour) in enumerate(zip(setups, IDENTITY, strict=True), start=1):
    values = offspring_cloud(make_operator(), PARENT_A, PARENT_B)[:, 0]
    centres, heights = density(values, 0, 10)
    fig.add_trace(
        go.Scatter(x=centres, y=heights, mode="lines", name=name, showlegend=False, line={"color": colour, "width": 2}),
        row=row,
        col=1,
    )
    mark_parents(fig, [3.0, 7.0], ("", ""), row=row, col=1)
    fig.update_yaxes(showticklabels=False, row=row, col=1)

style(
    fig, "Where each operator puts its offspring (parents at 3 and 7)", "value of x₁", "", height=760, showlegend=False
)
for annotation in fig.layout.annotations:
    annotation.update(font={"size": 12, "color": INK}, x=0, xanchor="left")
show(fig)

Reading the panels top to bottom:

- **Simulated binary** and **bounded exponential** put almost nothing between the
  parents. Their offspring cluster tightly on $3$ and $7$.
- **Blend alpha** and **local** do the opposite — they fill the interval. Local
  crossover never leaves it at all; blend alpha spills past both ends by a margin
  set by α.
- **Single arithmetic** produces three spikes, and the tallest is in the *middle*.
  With two variables it modifies $x_1$ only half the time: those offspring land
  exactly on the parent mean (about 51% of them), and the rest keep their own
  parent's $x_1$ untouched (about 24% on each parent).

The rest of this page takes each operator in turn.

## Simulated binary crossover (SBX)

SBX is the default for every continuous algorithm in DESDEO. It was designed to
reproduce, for real-valued variables, the *effect* single-point crossover has on
binary strings: offspring that mostly resemble one parent or the other, with the
occasional distant explorer.

It draws a **spread factor** $\beta$ from a polynomial distribution controlled by
the distribution index $\eta$:

$$
P(\beta) \;\propto\;
\begin{cases}
\beta^{\eta} & \beta \le 1 \quad \text{(contracting)}\\[4pt]
\beta^{-(\eta+2)} & \beta > 1 \quad \text{(expanding)}
\end{cases}
$$

Sampling it by inverting the CDF against a uniform $\mu$ gives the two lines that
do the real work in the implementation:

$$
\beta =
\begin{cases}
(2\mu)^{\frac{1}{\eta+1}} & \mu \le 0.5\\[4pt]
\bigl(2 - 2\mu\bigr)^{-\frac{1}{\eta+1}} & \mu > 0.5
\end{cases}
\qquad
c_{1,2} = \bar{x} \mp \beta\,\frac{x_1 - x_2}{2}
$$

where $\bar{x}$ is the parent mean. The density of $\beta$ peaks at $\beta = 1$,
and $\beta = 1$ means "land on a parent" — which is why SBX concentrates.

Note what the second equation says: SBX is built around the **parent mean**, and
the sign in front of $\beta$ decides which parent a given child is pulled toward.
That sign is a coin flip, and it is a knob —
see [per-variable mixing](#per-variable-mixing-the-uniform-crossover-layer) below.

References: Deb & Agrawal (1995), *Simulated binary crossover for continuous search
space*, Complex Systems 9(2); Deb & Gulati (2001), *Design of truss-structures for
minimum weight using genetic algorithms*, Finite Elements in Analysis and Design 37(5).

In [22]:
sbx = SimulatedBinaryCrossover(
    problem=PROBLEM,
    publisher=Publisher(),
    verbosity=1,
    seed=0,
    xover_distribution=15,  # eta: larger -> offspring hug the parents more tightly
    xover_probability=1.0,  # per-variable chance the variable is recombined at all
    bounded=False,  # see below
)

### η controls how far offspring stray

$\eta$ is the only parameter that changes the *shape* of the distribution. Small
$\eta$ produces a heavy tail — at $\eta = 2$ the offspring of parents sitting at
$3$ and $7$ can land a hundred units away. Large $\eta$ collapses the distribution
onto the parents.

In [23]:
fig = go.Figure()
for eta, colour in zip([2, 5, 15, 30], RAMP, strict=True):
    operator = SimulatedBinaryCrossover(
        problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, xover_distribution=eta
    )
    values = offspring_cloud(operator, PARENT_A, PARENT_B)[:, 0]
    centres, heights = density(values, -5, 15, bins=200)
    fig.add_trace(go.Scatter(x=centres, y=heights, mode="lines", name=f"η = {eta}", line={"color": colour, "width": 2}))
mark_parents(fig, [3.0, 7.0])
style(
    fig,
    "Simulated binary crossover: the effect of the distribution index η",
    "value of x₁  (window [-5, 15]; the η=2 tail runs far beyond it)",
    "density",
)
show(fig)

### The `bounded` variant

The equations above have no idea the problem has bounds, so a large $\beta$ can
push an offspring outside the feasible box. Setting `bounded=True` switches to the
truncated formulation of Deb & Gulati, which rescales $\beta$ per variable so that
offspring cannot escape.

Most other frameworks — pymoo, jMetalPy, Platypus, pagmo2, DEAP's
`cxSimulatedBinaryBounded`, and Deb's own NSGA-II C code — implement the bounded
variant while simply calling it "SBX". DESDEO gives you both and defaults to the
unbounded one, so this is worth setting deliberately.

Below, both parents sit near the lower bound, where the difference is stark.

In [24]:
near_bound_a, near_bound_b = [0.2, 0.2], [1.0, 1.0]
fig = make_subplots(
    rows=1, cols=2, column_widths=[0.68, 0.32], subplot_titles=("Offspring distribution", "Offspring outside [0, 10]")
)
escape_rates = []
for (label, is_bounded), colour in zip([("bounded=False", False), ("bounded=True", True)], IDENTITY, strict=False):
    operator = SimulatedBinaryCrossover(
        problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, xover_distribution=5, bounded=is_bounded
    )
    values = offspring_cloud(operator, near_bound_a, near_bound_b)[:, 0]
    centres, heights = density(values, -4, 5, bins=180)
    fig.add_trace(
        go.Scatter(x=centres, y=heights, mode="lines", name=label, line={"color": colour, "width": 2}), row=1, col=1
    )
    escape_rates.append((label, 100 * np.mean((values < 0) | (values > 10)), colour))

fig.add_vline(
    x=0.0,
    line={"color": INK, "width": 1.5},
    annotation={"text": "lower bound", "font": {"size": 10, "color": INK}},
    annotation_position="top left",
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=[label for label, _, _ in escape_rates],
        y=[rate for _, rate, _ in escape_rates],
        marker={"color": [colour for _, _, colour in escape_rates]},
        text=[f"{rate:.1f}%" for _, rate, _ in escape_rates],
        textposition="outside",
        textfont={"color": INK},
        showlegend=False,
    ),
    row=1,
    col=2,
)
style(fig, "Simulated binary crossover: bounded vs unbounded, parents at 0.2 and 1.0", "", "", height=420)
fig.update_xaxes(title_text="value of x₁", row=1, col=1)
fig.update_yaxes(title_text="density", row=1, col=1)
fig.update_yaxes(title_text="% of offspring", row=1, col=2, range=[0, 1.35 * max(rate for _, rate, _ in escape_rates)])
for annotation in fig.layout.annotations:
    annotation.update(font={"size": 12, "color": INK})
show(fig)

## Blend alpha crossover (BLX-α)

BLX-α takes the interval spanned by the two parents, widens it by a fraction α of
its own length at both ends, and samples uniformly from the result:

$$
c \sim U\bigl[\,c_{\min} - \alpha d,\; c_{\max} + \alpha d\,\bigr],
\qquad d = |x_1 - x_2|
$$

with the result clipped to the variable bounds. At $\alpha = 0$ offspring are
confined strictly between the parents; larger α lets them explore outside.

BLX-α is also the one operator that can produce more than two offspring per pair,
via `repeats`. It validates its inputs on construction: a negative `alpha`, or a
problem whose variables are not continuous, raises immediately.

Reference: Eshelman & Schaffer (1993), *Real-coded genetic algorithms and
interval-schemata*, FOGA 2.

In [25]:
blx = BlendAlphaCrossover(
    problem=PROBLEM,
    publisher=Publisher(),
    verbosity=1,
    seed=0,
    alpha=0.5,  # how far outside the parent interval offspring may land
    repeats=2,  # offspring generated per parent pair
    sample_each_component=True,  # fresh random weight per variable
)

In [26]:
fig = go.Figure()
for alpha, colour in zip([0.0, 0.25, 0.5, 1.0], RAMP, strict=True):
    operator = BlendAlphaCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, alpha=alpha)
    values = offspring_cloud(operator, PARENT_A, PARENT_B)[:, 0]
    centres, heights = density(values, 0, 10, bins=140)
    fig.add_trace(
        go.Scatter(x=centres, y=heights, mode="lines", name=f"α = {alpha}", line={"color": colour, "width": 2})
    )
mark_parents(fig, [3.0, 7.0])
style(fig, "Blend alpha crossover: α widens the sampling interval", "value of x₁", "density")
show(fig)

Each α gives a flat-topped distribution — uniform, as advertised — whose width
grows with α. At $\alpha = 0$ the support is exactly $[3, 7]$.

## Local crossover

Local crossover is the plain arithmetic recombination: a random convex combination
of the two parents, with a **fresh weight drawn for every decision variable**.

$$
c_1 = \alpha x_1 + (1 - \alpha) x_2, \qquad
c_2 = (1 - \alpha) x_1 + \alpha x_2, \qquad \alpha \sim U[0, 1]
$$

Because α is a convex weight, offspring can never leave the segment between the
parents — the safest of the continuous operators, and the least exploratory. It
takes no parameters at all.

Reference: Dumitrescu, Lazzerini, Jain & Dumitrescu (2000), *Evolutionary
Computation*, CRC Press.

In [27]:
local = LocalCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0)

## Single arithmetic crossover

The most conservative operator here. It picks **one** variable position $k$ at
random and replaces it, in both offspring, with the mean of the parents at that
position. Every other variable is copied unchanged:

$$
c_{1,k} = c_{2,k} = \frac{x_{1,k} + x_{2,k}}{2},
\qquad c_{i,j} = x_{i,j} \;\; \text{for } j \neq k
$$

`xover_probability` is the chance that a given *pair* is recombined at all; pairs
that fail the roll are passed through untouched.

Note that $k$ is drawn per pair, so on a two-variable problem $x_1$ is the chosen
position only half the time. That is what produces the three spikes in the overview
figure: about half the offspring sit on the parent mean, and the remaining half
keep their own parent's $x_1$ exactly, split evenly between the two parents.

Reference: Picek, Jakobovic & Golub (2013), CEC.

In [28]:
single_arithmetic = SingleArithmeticCrossover(
    problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, xover_probability=1.0
)

## Bounded exponential crossover (BEX)

BEX is **parent-centric** in the strictest sense: each offspring is a displacement
away from *its own* parent, not from the parent mean.

$$
c = x + \beta\,|x_1 - x_2|
$$

where $\beta$ comes from inverting the CDF of an exponential distribution that has
been **truncated at the variable bounds** — one branch toward the lower bound, one
toward the upper, chosen by a coin flip:

$$
\beta =
\begin{cases}
\lambda \log\!\bigl(e_{\text{lo}} + u\,(1 - e_{\text{lo}})\bigr), & r \le 0.5\\[4pt]
-\lambda \log\!\bigl(1 - u\,(1 - e_{\text{hi}})\bigr), & r > 0.5
\end{cases}
\qquad
\begin{aligned}
e_{\text{lo}} &= \exp\!\Bigl(\tfrac{x^{\text{lo}} - x}{\lambda\,|x_1 - x_2|}\Bigr)\\
e_{\text{hi}} &= \exp\!\Bigl(\tfrac{x - x^{\text{up}}}{\lambda\,|x_1 - x_2|}\Bigr)
\end{aligned}
$$

Because the truncation is built into the sampling rather than bolted on afterwards,
offspring are inside the bounds by construction — no clipping, and no pile-up at
the bound.

λ scales the exponential relative to the distance between the parents. It changes
the *spread* only: the median offspring sits on its own parent for every λ.

References: Thakur, Meghwani & Jalota (2014), *A modified real coded genetic
algorithm for constrained optimization*, Applied Mathematics and Computation 235;
modifying the Laplace crossover of Deep & Thakur (2007).

In [29]:
bex = BoundedExponentialCrossover(
    problem=PROBLEM,
    publisher=Publisher(),
    verbosity=1,
    seed=0,
    lambda_=0.25,  # spread of the exponential, relative to the parent distance
    xover_probability=1.0,  # per-pair chance of recombining at all
)

In [30]:
fig = go.Figure()
for lam, colour in zip([0.1, 0.25, 0.5, 2.0], RAMP, strict=True):
    operator = BoundedExponentialCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, lambda_=lam)
    values = offspring_cloud(operator, PARENT_A, PARENT_B)[:, 0]
    centres, heights = density(values, 0, 10, bins=180)
    fig.add_trace(go.Scatter(x=centres, y=heights, mode="lines", name=f"λ = {lam}", line={"color": colour, "width": 2}))
mark_parents(fig, [3.0, 7.0])
style(fig, "Bounded exponential crossover: λ sets the spread, not the centre", "value of x₁", "density")
show(fig)

Every curve peaks on the parents and decays away from them. Increasing λ flattens
the distribution toward uniform over the whole feasible range, but never shifts the
peaks — contrast this with SBX, where η changes the tail of a *polynomial* density
around the parent mean, and where offspring essentially never land at the midpoint.

## Per-variable mixing: the uniform-crossover layer

Several continuous operators carry a second, separate mechanism inside them: a
**per-variable choice** that is applied on top of the sampling equations above.
It is easy to miss, because it is almost invisible in the pooled histograms — but
it changes the behaviour substantially.

It shows up in three operators:

- **SBX** flips the sign of β per variable with probability
  `uniform_xover_probability`, which decides *which* parent that gene is drawn
  toward. As the source comment puts it, this makes SBX
  *"more similar to uniform crossover than single-point crossover"*.
- **SBX** also has `xover_probability`, a per-variable gate that skips
  recombination entirely and copies the gene verbatim.
- **BLX-α** exposes `sample_each_component`: one random weight per variable, or a
  single weight reused across the whole offspring vector.
- **Local crossover** always draws a fresh weight per variable, with no toggle.

### `uniform_xover_probability`: which parent a child takes after

To see this at all, the offspring have to be split by **which parent they descend
from** — pooled together, the two streams cancel out and the marginal looks
symmetric no matter what this parameter is set to.

In [31]:
fig = make_subplots(
    rows=1,
    cols=4,
    shared_yaxes=True,
    horizontal_spacing=0.02,
    subplot_titles=[f"u = {u}" for u in (0.0, 0.25, 0.5, 1.0)],
)
for column, u in enumerate([0.0, 0.25, 0.5, 1.0], start=1):
    operator = SimulatedBinaryCrossover(
        problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, xover_distribution=5, uniform_xover_probability=u
    )
    offspring = offspring_cloud(operator, PARENT_A, PARENT_B)
    # The unbounded SBX path writes offspring[i] and offspring[i+1] inside its loop,
    # so children are interleaved: even rows descend from parent A, odd rows from B.
    for values, label, colour in (
        (offspring[0::2, 0], "children of parent A (at 3)", IDENTITY[0]),
        (offspring[1::2, 0], "children of parent B (at 7)", IDENTITY[1]),
    ):
        centres, heights = density(values, 0, 10, bins=120)
        fig.add_trace(
            go.Scatter(
                x=centres,
                y=heights,
                mode="lines",
                name=label,
                legendgroup=label,
                showlegend=(column == 1),
                line={"color": colour, "width": 2},
            ),
            row=1,
            col=column,
        )
    mark_parents(fig, [3.0, 7.0], ("", ""), row=1, col=column)
    fig.update_yaxes(showticklabels=False, row=1, col=column)

style(fig, "SBX: uniform_xover_probability decides which parent each gene is pulled toward", "", "", height=400)
fig.update_xaxes(title_text="value of x₁")
for annotation in fig.layout.annotations:
    annotation.update(font={"size": 12, "color": INK})
show(fig)

At $u = 0$ each child stays on its own parent — no mixing at all. At $u = 1$ the
roles are swapped wholesale, and every child takes after the *other* parent. At
$u = 0.5$, the default, the two streams overlap exactly, and it is only their sum
that looks like the familiar symmetric SBX picture.

### `xover_probability`: how many genes get recombined at all

This is a different knob. It is a per-variable gate applied *before* the mixing
above: genes that fail the roll are copied from the parent untouched.

In [32]:
probabilities = np.linspace(0, 1, 11)
unchanged = []
for p in probabilities:
    operator = SimulatedBinaryCrossover(
        problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, xover_distribution=15, xover_probability=float(p)
    )
    offspring = offspring_cloud(operator, PARENT_A, PARENT_B, n_pairs=4000)
    unchanged.append(100 * np.isclose(offspring[0::2], 3.0).mean())

fig = go.Figure(
    go.Scatter(
        x=probabilities,
        y=unchanged,
        mode="lines+markers",
        name="genes copied verbatim",
        line={"color": IDENTITY[0], "width": 2},
        marker={"size": 8},
    )
)
style(
    fig,
    "SBX: xover_probability gates how many genes are recombined at all",
    "xover_probability",
    "% of genes identical to the parent",
    showlegend=False,
)
show(fig)

The relationship is exactly linear, as it should be: `xover_probability` is a
per-variable Bernoulli trial. At $0$ the operator is the identity function; at $1$
every gene is recombined.

### `sample_each_component` in BLX-α

The same idea in a different operator, and the clearest way to see it is in two
dimensions. With one shared random weight the offspring collapse onto a single
line through the parent box; with a fresh weight per variable they fill it.

In [33]:
corner_a, corner_b = [2.0, 8.0], [8.0, 2.0]
fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.06,
    subplot_titles=("sample_each_component=False", "sample_each_component=True"),
)
for column, per_component in enumerate([False, True], start=1):
    operator = BlendAlphaCrossover(
        problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, alpha=0.5, sample_each_component=per_component
    )
    offspring = offspring_cloud(operator, corner_a, corner_b, n_pairs=1200)
    fig.add_trace(
        go.Scatter(
            x=offspring[:, 0],
            y=offspring[:, 1],
            mode="markers",
            showlegend=False,
            marker={"size": 4, "color": IDENTITY[0], "opacity": 0.35},
        ),
        row=1,
        col=column,
    )
    fig.add_trace(
        go.Scatter(
            x=[corner_a[0], corner_b[0]],
            y=[corner_a[1], corner_b[1]],
            mode="markers+text",
            text=["parent A", "parent B"],
            textposition="top center",
            textfont={"color": INK, "size": 10},
            showlegend=False,
            marker={"size": 11, "color": INK, "symbol": "diamond"},
        ),
        row=1,
        col=column,
    )
style(fig, "Blend alpha crossover: one shared weight vs one weight per variable", "", "", height=430, showlegend=False)
fig.update_xaxes(title_text="x₁", range=[0, 10])
fig.update_yaxes(title_text="x₂", range=[0, 10], row=1, col=1)
for annotation in fig.layout.annotations:
    annotation.update(font={"size": 12, "color": INK})
show(fig)

## Operators for discrete variables

Three operators handle non-continuous variables. They share the same skeleton and
differ only in *how the genes are divided* between the parents, and in how the
values are cast.

- **`SinglePointBinaryCrossover`** — binary variables. One cut point is drawn per
  pair, and the two parents swap everything after it. Because the cut is a single
  position, neighbouring genes tend to be inherited together. Needs at least two
  variables, and raises otherwise. *Holland (1975); Goldberg (1989).*
- **`UniformIntegerCrossover`** — integer variables. Every gene is independently
  taken from one parent or the other, using a mask drawn fresh for each pair.
  *Syswerda (1989).*
- **`UniformMixedIntegerCrossover`** — identical to the above, but values are kept
  as floats so continuous components are not rounded. This is the default for the
  mixed-integer algorithms.

None of the three takes any parameters.

In [34]:
binary_problem = demo_problem(10, "binary", 0, 1)
single_point = SinglePointBinaryCrossover(problem=binary_problem, publisher=Publisher(), verbosity=1, seed=0)

Mating an all-zeros parent with an all-ones parent makes the inheritance pattern
directly readable: for each gene position, the average offspring value *is* the
probability that the gene came from the all-ones parent.

In [35]:
zeros, ones = [0.0] * 10, [1.0] * 10
integer_problem = demo_problem(10, "integer", 0, 1)
fig = go.Figure()
for (label, problem, operator), colour in zip(
    [
        (
            "Single point binary",
            binary_problem,
            SinglePointBinaryCrossover(problem=binary_problem, publisher=Publisher(), verbosity=1, seed=0),
        ),
        (
            "Uniform integer",
            integer_problem,
            UniformIntegerCrossover(problem=integer_problem, publisher=Publisher(), verbosity=1, seed=0),
        ),
    ],
    IDENTITY,
    strict=False,
):
    n_pairs = 4000
    offspring = offspring_cloud(operator, zeros, ones, n_pairs=n_pairs, problem=problem)
    # These operators stack all first-children before all second-children.
    profile = offspring[:n_pairs].mean(axis=0)
    fig.add_trace(
        go.Scatter(
            x=np.arange(1, 11),
            y=profile,
            mode="lines+markers",
            name=label,
            line={"color": colour, "width": 2},
            marker={"size": 8},
        )
    )

fig.add_hline(y=0.5, line={"color": INK_SOFT, "width": 1, "dash": "dot"})
style(
    fig, "Inheritance profile: chance each gene comes from the second parent", "variable index", "P(gene from parent B)"
)
fig.update_yaxes(range=[-0.05, 1.05])
fig.update_xaxes(dtick=1)
show(fig)

The two curves say everything about the difference. **Uniform crossover** is flat
at $0.5$ — every gene is an independent coin flip, so position carries no
information. **Single-point crossover** ramps from $0$ to $1$ across the string:
the first gene almost always comes from the first parent and the last almost
always from the second, because a single cut point cannot separate the ends.

That ramp is also why single-point crossover preserves contiguous blocks of genes,
and uniform crossover deliberately destroys them.

## Composite crossover

`CompositeCrossover` wraps a list of operators and delegates to them in
**round-robin**: the first call uses the first operator, the second call the
second, and so on. It is the way to combine operators when different generations —
or different variable groups — call for different recombination behaviour.

In [36]:
composite = CompositeCrossover(
    problem=PROBLEM,
    publisher=Publisher(),
    verbosity=1,
    seed=0,
    operators=[
        SimulatedBinaryCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, xover_distribution=15),
        BlendAlphaCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, alpha=0.5),
    ],
)

## Geometry in two dimensions

The one-dimensional densities hide something: whether an operator treats the
variables independently. Here the parents sit at opposite corners of the box, and
each panel shows 1200 offspring.

In [37]:
geometry = [
    (
        "Simulated binary (η=15)",
        SimulatedBinaryCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, xover_distribution=15),
    ),
    (
        "Bounded exponential (λ=0.25)",
        BoundedExponentialCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, lambda_=0.25),
    ),
    (
        "Blend alpha (α=0.5)",
        BlendAlphaCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, alpha=0.5),
    ),
    ("Local", LocalCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0)),
    (
        "Single arithmetic",
        SingleArithmeticCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=0, xover_probability=1.0),
    ),
]

fig = make_subplots(
    rows=2,
    cols=3,
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.04,
    vertical_spacing=0.1,
    subplot_titles=[name for name, _ in geometry] + [""],
)
for index, ((_name, operator), colour) in enumerate(zip(geometry, IDENTITY, strict=True)):
    row, column = divmod(index, 3)
    offspring = offspring_cloud(operator, corner_a, corner_b, n_pairs=1200)
    fig.add_trace(
        go.Scatter(
            x=offspring[:, 0],
            y=offspring[:, 1],
            mode="markers",
            showlegend=False,
            marker={"size": 3.5, "color": colour, "opacity": 0.35},
        ),
        row=row + 1,
        col=column + 1,
    )
    fig.add_trace(
        go.Scatter(
            x=[corner_a[0], corner_b[0]],
            y=[corner_a[1], corner_b[1]],
            mode="markers",
            showlegend=False,
            marker={"size": 9, "color": INK, "symbol": "diamond"},
        ),
        row=row + 1,
        col=column + 1,
    )
style(fig, "Offspring geometry: parents at opposite corners (♦) of [0, 10]²", "", "", height=620, showlegend=False)
fig.update_xaxes(title_text="x₁", range=[0, 10])
fig.update_yaxes(title_text="x₂", range=[0, 10])
for annotation in fig.layout.annotations:
    annotation.update(font={"size": 12, "color": INK})
show(fig)

- **Single arithmetic** collapses to just **four points**. It replaces exactly one
  coordinate with the parent mean and copies the other, so with two variables the
  only reachable offspring are each parent with one of its coordinates moved to the
  centre.
- **Local** fills the box spanned by the parents, and fills it evenly — not the
  diagonal between them. Its weight is drawn afresh for each variable, so $x_1$ and
  $x_2$ are mixed independently and the offspring are uncorrelated. A single shared
  weight would give the diagonal line instead.
- **Blend alpha** fills the same kind of box, just a wider one: α pushes the edges
  out beyond the parents, up to the variable bounds.
- **BEX** stays on the two parent corners and essentially never reaches the other
  two — under 1% of its offspring land there.
- **SBX** spreads over all **four** corners in roughly equal proportion, the two
  parent corners and the two opposite ones alike. Its per-variable sign flip takes
  $x_1$ from one parent and $x_2$ from the other as readily as not, so it
  recombines the coordinates rather than preserving the parent's combination of
  them. This is the uniform-crossover layer showing up as geometry, and it is the
  sharpest difference between the two concentrating operators.

## From a pair to a population

Everything so far fixed one pair of parents to isolate the operator's sampling
distribution. In an actual generation the parents are spread out, and the operator
is applied across the whole mating pool.

In [38]:
publisher = Publisher()
evaluator = EMOEvaluator(problem=PROBLEM, publisher=publisher, verbosity=1)
generator = RandomGenerator(
    problem=PROBLEM, evaluator=evaluator, publisher=publisher, n_points=200, seed=3, verbosity=1
)
parents, _ = generator.do()

population_ops = [
    (
        "Simulated binary (η=15)",
        SimulatedBinaryCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=1, xover_distribution=15),
    ),
    (
        "Blend alpha (α=0.5)",
        BlendAlphaCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=1, alpha=0.5),
    ),
    ("Local", LocalCrossover(problem=PROBLEM, publisher=Publisher(), verbosity=1, seed=1)),
]
parent_values = parents[["x_1", "x_2"]].to_numpy()

fig = make_subplots(
    rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.04, subplot_titles=[name for name, _ in population_ops]
)
for column, ((_name, operator), colour) in enumerate(
    zip(population_ops, IDENTITY[: len(population_ops)], strict=True), 1
):
    offspring = operator.do(population=parents).to_numpy()
    fig.add_trace(
        go.Scatter(
            x=parent_values[:, 0],
            y=parent_values[:, 1],
            mode="markers",
            name="parents",
            legendgroup="parents",
            showlegend=(column == 1),
            marker={"size": 6, "color": INK_SOFT, "opacity": 0.45},
        ),
        row=1,
        col=column,
    )
    fig.add_trace(
        go.Scatter(
            x=offspring[:, 0],
            y=offspring[:, 1],
            mode="markers",
            name="offspring",
            legendgroup="offspring",
            showlegend=(column == 1),
            marker={"size": 6, "color": colour, "opacity": 0.65},
        ),
        row=1,
        col=column,
    )
style(fig, "One generation: 200 random parents and the offspring they produce", "", "", height=430)
fig.update_xaxes(title_text="x₁", range=[-0.5, 10.5])
fig.update_yaxes(title_text="x₂", range=[-0.5, 10.5], row=1, col=1)
for annotation in fig.layout.annotations:
    annotation.update(font={"size": 12, "color": INK})
show(fig)

Local crossover visibly contracts the population toward its centre — every
offspring is a convex combination of two parents, so the cloud can only ever
shrink (here the mean standard deviation falls from about 2.85 to 2.26 in a single
generation). SBX holds the spread almost exactly, and blend alpha loses only a
little. That is a good reason to prefer them as defaults: an operator that
contracts on its own leaves all of the exploring to mutation.

## Choosing an operator

| If your variables are… | Use |
|---|---|
| continuous | `SimulatedBinaryCrossover` — the default for RVEA, NSGA-III and SMS-EMOA |
| binary | `SinglePointBinaryCrossover` |
| integer | `UniformIntegerCrossover` |
| mixed | `UniformMixedIntegerCrossover` — the default for the mixed-integer algorithms |

Within the continuous operators, the choice is about how much exploration you
want. `LocalCrossover` and `SingleArithmeticCrossover` are conservative and can
only contract the population. `BlendAlphaCrossover` with a moderate α, and SBX with
a small η, explore aggressively. `BoundedExponentialCrossover` is a good middle
ground when respecting bounds matters and you do not want offspring piling up
against them.

One practical note: if you use SBX on a problem with meaningful bounds, consider
`bounded=True`. DESDEO defaults to the unbounded formulation, unlike most other
frameworks.

Operators can also be selected declaratively, without importing them — see the
[Pydantic interface](../pydantic_interface):

```python
from desdeo.emo.options import crossover

options.template.crossover = crossover.BlendAlphaCrossoverOptions(alpha=0.5)
```

## Implementation details

Everything below is shared by all nine operators, and is the reason they are
interchangeable.

**Construction.** Every operator subclasses `BaseCrossover` and is built the same
way:

```python
operator = SomeCrossover(problem=problem, publisher=publisher, verbosity=1, seed=0, ...)
```

Bounds and types are read from `problem.get_flattened_variables()`, so
`TensorVariable`s are handled transparently as flat columns.

**Randomness.** Each operator owns a private `numpy` generator seeded from `seed`.
There is no shared global random state, so an operator's output depends only on its
own seed and call history.

**The `do` method.** Keyword-only, and the only method an algorithm calls:

```python
offspring = operator.do(population=population, to_mate=None)
```

It returns a `polars.DataFrame` containing **only the decision-variable columns** —
objectives and constraints are not carried through.

**Pairing.** With `to_mate=None` the whole population is shuffled and paired up.
With an explicit `to_mate`, the order is used verbatim, so the caller controls
exactly who mates with whom. Pairing is always consecutive: entries $(0,1)$,
$(2,3)$, and so on. An odd-length mating pool is padded by repeating its first
member, and the resulting extra offspring is discarded — which is why the count
always comes out at one offspring per mated parent.

**Messages.** Every crossover operator is a `Subscriber` that only ever publishes;
none of them subscribes to anything. What is published depends on `verbosity`:

| verbosity | published |
|---|---|
| 0 | nothing |
| 1 | the operator's scalar parameters (`XOVER_PROBABILITY`, `XOVER_DISTRIBUTION`, `ALPHA`, `LAMBDA`) |
| 2 | the above plus the `PARENTS` and `OFFSPRINGS` dataframes |

Dataframe messages are restricted to verbosity 2 because the message layer only
admits scalar payloads at verbosity 1.